In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from matplotlib.ticker import ScalarFormatter

import matplotlib.colors as colors
import os
import scipy.stats as stats
import sys
import seaborn as sns

# Specify the path for missense variant data 
var_dir = '/media/drive2/seulki/Project/Missense_2025/data/variant-2025Q4'

# Datasets from src/statistical_analysis/1_enrichment_calculation.ipynb

In [10]:
df_case = pd.read_csv(f'{var_dir}/final_case_annotation.txt',delimiter='\t')
df_case

,Unnamed: 0,Gene,UniProt,UniProt_Isoform,Type,ResID,RefAA,AltAA,Source,Transcript,...,DS_inter,NB_inter,gnomAD_presence,PFES,PFES_Physicochemical,PFES_Function,PFES_Domain,PFES_Modification,PFES_Structure,PFES_PPI
0,1,A2ML1,A8K2U0,A8K2U0-1,Missense,356,P,R,HGMD,NM_144670.6,...,-,-,False,2.46,1.59,NaN,NaN,NaN,0.87,NaN
1,7,A2ML1,A8K2U0,A8K2U0-1,Missense,822,V,I,HGMD,NM_144670.6,...,-,-,False,4.07,-2.11,NaN,NaN,NaN,6.19,NaN
2,17,A4GALT,Q9NPC4,Q9NPC4,Missense,211,Q,E,ClinVar;HGMD,NM_017436.7;NM_017436.7,...,-,-,False,3.52,-1.41,NaN,0.01,NaN,4.91,NaN
3,21,AAAS,Q9NRG9,Q9NRG9-1,Missense,1,M,V,ClinVar,NM_015665.6,...,-,-,False,-5.25,-1.89,NaN,NaN,NaN,-3.36,NaN
4,25,AAAS,Q9NRG9,Q9NRG9-1,Missense,41,Q,R,HGMD,NM_015665.6,...,-,-,False,-2.51,-1.68,NaN,-0.51,NaN,-0.32,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85316,137229,ZP3,P21754,P21754-1,Missense,387,V,M,HGMD,NM_001110354.2,...,-,-,False,1.31,-1.89,NaN,2.02,NaN,1.18,NaN
85317,137230,ZP4,Q12836,Q12836,Missense,447,C,Y,HGMD,NM_021186.5,...,-,-,False,17.02,3.63,NaN,2.03,NaN,11.36,NaN
85318,137232,ZPBP,Q9BS86,Q9BS86-1,Missense,217,H,L,HGMD,NM_007009.3,...,-,-,False,5.20,0.29,NaN,-0.51,NaN,5.42,NaN
85319,137235,ZSWIM6,Q9HCJ5,Q9HCJ5,Missense,1163,R,W,ClinVar;HGMD,NM_020928.2;NM_020928.2,...,-,-,False,-0.64,0.47,NaN,-0.51,NaN,-0.60,NaN


In [11]:
df_control = pd.read_csv(f'{var_dir}/final_control_annotation.txt',delimiter='\t')
df_control

/tmp/ipykernel_1389566/3782279739.py:1: DtypeWarning: Columns (14,66,67,68,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df_control = pd.read_csv(f'{var_dir}/final_control_annotation.txt',delimiter='\t')


,Unnamed: 0,Gene,UniProt,UniProt_Isoform,Source,Type,ResID,RefAA,AltAA,Change_nucleotide,...,PFES,PFES_Physicochemical,PFES_Function,PFES_Domain,PFES_Modification,PFES_Structure,PFES_PPI,Phenotype,CS_Confidence,gnomAD_presence
0,61,A1BG,P04217,P04217-1,gnomAD,Missense,52,H,R,c.155A>G,...,1.64,-1.39,NaN,0.92,NaN,2.11,NaN,NaN,NaN,NaN
1,2084,A2M,P01023,P01023,gnomAD,Missense,639,N,D,c.1915A>G,...,5.17,-1.31,NaN,NaN,NaN,1.69,4.8,NaN,NaN,NaN
2,2300,A2M,P01023,P01023,gnomAD,Missense,844,A,V,c.2531C>T,...,-3.15,-1.17,NaN,NaN,NaN,-1.98,NaN,NaN,NaN,NaN
3,2459,A2M,P01023,P01023,gnomAD,Missense,1000,I,V,c.2998A>G,...,1.92,-2.11,NaN,NaN,NaN,4.03,NaN,NaN,NaN,NaN
4,3920,A2ML1,A8K2U0,A8K2U0-1,gnomAD,Missense,850,D,E,c.2550C>A,...,1.94,-2.33,NaN,NaN,NaN,4.27,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130827,109238,ZZEF1,O43149,O43149-1,ClinVar,Missense,2525,K,R,c.7574A>G,...,-2.73,-2.22,NaN,-0.51,NaN,NaN,NaN,not specified,BLB,True
130828,109239,ZZZ3,Q8IYH5,Q8IYH5-1,ClinVar,Missense,251,M,V,c.751A>G,...,-7.12,-1.86,NaN,NaN,NaN,-5.26,NaN,not specified,BLB,True
130829,109240,ZZZ3,Q8IYH5,Q8IYH5-1,ClinVar,Missense,336,N,S,c.1007A>G,...,-11.33,-2.78,NaN,-3.44,NaN,-5.11,NaN,not specified,BLB,True
130830,109241,ZZZ3,Q8IYH5,Q8IYH5-1,ClinVar,Missense,425,Q,E,c.1273C>G,...,-10.17,-1.47,NaN,-3.44,NaN,-5.26,NaN,not specified,BLB,True


# Dump to data file for repository only relevant columns

In [15]:
dump_cols = ['Gene', 'UniProt', 'UniProt_Isoform', 'Type', 'ResID',
       'RefAA', 'AltAA', 'Source', 'Transcript', 'Change_nucleotide',
       'Phenotype', 'CS_Confidence', 'protein_class', 'phys_ref', 'AApchem',
       'AAdistance', 'ss3', 'ss9', 'plddt', 'phi', 'psi', 'asa', 'rsa',
       'Active site', 'Binding site', 'Site', 'DNA binding region',
       'Zinc finger', 'Region', 'Region/Disordered', 'Region/Interaction',
       'Region/Others', 'Motif', 'Coiled coil', 'Compositional bias', 'Repeat',
       'Domain', 'Topological domain', 'Transmembrane', 'Intramembrane',
       'Signal peptide', 'Transit peptide', 'Propeptide', 'Peptide', 'Chain',
       'Lipidation', 'Glycosylation', 'Crosslinks', 'Modified residue',
       'Acetylation', 'Methylation', 'Phosphorylation', 'SUMOylation',
       'Ubiquitination', 'O-GalNAc/GlcNAc', 'HB_intra', 'SB_intra', 'DS_intra',
       'NB_intra', 'HB_inter', 'SB_inter', 'DS_inter', 'NB_inter']


In [23]:
df_case[dump_cols].to_csv('../data/case_annotation.tsv', sep='\t', index=False)
df_control[dump_cols].to_csv('../data/control_annotation.tsv', sep='\t', index=False)

print (f'File generated: ../data/case_annotation.tsv, ../data/control_annotation.tsv')

File generated: ../data/case_annotation.tsv, ../data/control_annotation.tsv
